In [9]:
import numpy as np

def make_data(n=200, noise=0.5, seed=0):
    """Known w and b, and deliberately unscaled columns (stds ~1, 10, 0.1)."""

    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, 3)) * np.array([1.0, 10.0,0.1])
    w_true,b_true = np.array([2.0, -3.0, 0.5]), 4.0
    y = X @ w_true + b_true + rng.normal(scale=noise, size=n)
    return X, y, w_true, b_true


def fit_normal_equation(X, y):
    """Solve (X'X theta = X'y directly. Bias = a column of ones"""
    Xb = np.hstack([np.ones((X.shape[0], 1)), X])
    theta = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)
    return theta[1:], theta[0]

def fit_gradient_descent(X, y, lr=0.1, epochs=500):
    """Batch gradient descent on MSE. Bias kept out of X so both gradients stay readable"""
    n, d = X.shape
    w, b, history = np.ones(d), 0.0, []
    for _ in range(epochs):
        resid = X @ w + b - y
        history.append(np.mean(resid**2))
        w -= lr * (2/n) * (X.T @ resid)
        b -= lr * (2/n) * resid.sum()
    return w, b, np.array(history)

def standardize(X):
    mu, sigma = X.mean(axis=0), X.std(axis=0)
    return (X - mu) / sigma, mu, sigma

if __name__ == "__main__":
    X, y, w_true, b_true = make_data()

    # 1. closed form on raw features
    w_ne, b_ne = fit_normal_equation(X, y)

    # 2. gradient descent on standardized features, then map back to raw units
    Xs, mu, sigma = standardize(X)
    w_s, b_s, hist = fit_gradient_descent(Xs, y, lr=0.1, epochs=500)
    w_gd = w_s / sigma
    b_gd = b_s - (mu / sigma) @ w_s

    print("true       ", np.round(np.append(w_true, b_true), 4))
    print("normal equation", np.round(np.append(w_ne, b_ne), 4))
    print("gradient desc", np.round(np.append(w_gd, b_gd), 4))
    print("MSE first epoch", round(float(hist[0]), 4), "| last epoch", round(float(hist[-1]), 4))

    # 3. same lr unscaled features: the step that works above now explodes

    with np.errstate(over="ignore", invalid="ignore"):
        _, _, hist_raw = fit_gradient_descent(X, y, lr=0.1, epochs=500)
        print("unscaled, same lr -> final MSE", hist_raw[-1])

        # 4. diff against the reference
        from sklearn.linear_model import LinearRegression
        sk = LinearRegression().fit(X,y)
        assert np.allclose(w_ne, sk.coef_, atol=1e-8), "closed form must be exact"
        assert np.isclose(b_ne, sk.intercept_, atol=1e-8)
        assert np.allclose(w_gd, sk.coef_, atol=1e-8), "GD only converges towards it"
        print("matches sklearn")




true        [ 2.  -3.   0.5  4. ]
normal equation [ 1.955  -3.0048  0.7651  3.9819]
gradient desc [ 1.955  -3.0048  0.7651  3.9819]
MSE first epoch 969.3812 | last epoch 0.25
unscaled, same lr -> final MSE nan
matches sklearn
